## Visualize prototype space (2D)\n\nThis notebook reads artifacts created by `embed_space.ipynb` under `artifacts/embed_space/` and shows:\n- 2D projection of interaction (text+image) embeddings\n- Prototype vectors (one per inferred rating value)\n- Which points are closest to which prototype (optional connectors)\n- Closeness summaries (cosine similarity distributions)\n

In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120

In [ ]:
# ---- Paths (portable) ----
# By default, assume this notebook is run from the same folder as the artifacts.
# If you run elsewhere (e.g., Colab), set ROOT_OVERRIDE to the mounted project folder.
ROOT_OVERRIDE = None  # e.g. r"/content/drive/MyDrive/DIscussion_14-04-2026" or r"E:\\Meta D\\MMRecSys\\DIscussion_14-04-2026"

ROOT = Path(ROOT_OVERRIDE) if ROOT_OVERRIDE else Path.cwd()
ARTIFACT_DIR = ROOT / "artifacts" / "embed_space"

prototypes_path = ARTIFACT_DIR / "prototypes.npz"
predictions_path = ARTIFACT_DIR / "predictions.csv"
embeddings_path = ARTIFACT_DIR / "pair_embeddings.npz"

prototypes_path.exists(), predictions_path.exists(), embeddings_path.exists()

### 1) Load artifacts

In [ ]:
proto_npz = np.load(prototypes_path, allow_pickle=False)
proto_ratings = proto_npz["proto_ratings"].astype(np.float32)
proto_counts = proto_npz["proto_counts"].astype(np.int64)
proto_vectors = proto_npz["proto_vectors"].astype(np.float32)  # [K, D]

pred_df = pd.read_csv(predictions_path)

emb_npz = np.load(embeddings_path, allow_pickle=False)
E = emb_npz["pair_embeddings"].astype(np.float32)  # [N, D]
true_rating = emb_npz["true_rating"].astype(np.float32)
pred_rating = emb_npz["predicted_rating"].astype(np.float32)
top_sim = emb_npz["top_cosine_similarity"].astype(np.float32)

K, D = proto_vectors.shape
N = E.shape[0]
assert E.shape[1] == D

print(f"N={N}, D={D}, K={K}, ratings={proto_ratings.tolist()}")
pred_df.head()

### 2) PCA to 2D (deterministic)

We fit PCA on the union of interaction embeddings and prototype vectors, then project both.

In [ ]:
def pca_2d(X: np.ndarray) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
    """Return 2D PCA projection and model components.

    Uses SVD on mean-centered data for deterministic results.
    """
    X = X.astype(np.float64, copy=False)
    mu = X.mean(axis=0, keepdims=True)
    Xc = X - mu
    # economy SVD
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    W = Vt[:2].T  # [D,2]
    Z = Xc @ W
    return Z.astype(np.float32), {"mean": mu.astype(np.float32), "W": W.astype(np.float32), "S": S[:2].astype(np.float32)}


X_all = np.concatenate([E, proto_vectors], axis=0)
Z_all, pca_model = pca_2d(X_all)

Z_E = Z_all[:N]
Z_P = Z_all[N:]

Z_E.shape, Z_P.shape

### 3) 2D plot (points + prototypes + optional connectors)

In [ ]:
def rating_palette(values):
    # deterministic palette for up to ~10 discrete values
    base = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    ]
    vals = list(values)
    return {v: base[i % len(base)] for i, v in enumerate(vals)}


distinct_true = sorted(set(float(x) for x in true_rating.tolist()))
pal = rating_palette(distinct_true)

max_points = 2000  # downsample for readability if needed
if N > max_points:
    rng = np.random.default_rng(0)
    idx = rng.choice(N, size=max_points, replace=False)
else:
    idx = np.arange(N)

fig, ax = plt.subplots(figsize=(10, 7))

# Points colored by true rating
colors = [pal[float(true_rating[i])] for i in idx]
sc = ax.scatter(Z_E[idx, 0], Z_E[idx, 1], s=18, c=colors, alpha=0.55, linewidths=0)

# Prototypes as larger markers
ax.scatter(Z_P[:, 0], Z_P[:, 1], s=260, c="black", marker="X", alpha=0.9)

for k, r in enumerate(proto_ratings.tolist()):
    ax.text(
        Z_P[k, 0],
        Z_P[k, 1],
        f"rating={int(r) if float(r).is_integer() else r}\n(n={int(proto_counts[k])})",
        fontsize=9,
        ha="left",
        va="bottom",
        color="black",
    )

# Optional connectors: for a small subset, draw line to nearest prototype by cosine similarity
connect_n = 120
if N > 0 and connect_n > 0:
    # Use the already-computed top match stored in predictions
    # Map predicted rating -> prototype index
    rating_to_k = {float(proto_ratings[k]): k for k in range(len(proto_ratings))}
    pick = idx[: min(connect_n, len(idx))]
    for i in pick:
        k = rating_to_k[float(pred_rating[i])]
        ax.plot(
            [Z_E[i, 0], Z_P[k, 0]],
            [Z_E[i, 1], Z_P[k, 1]],
            color=pal[float(true_rating[i])],
            alpha=0.15,
            linewidth=1,
        )

ax.set_title("Prototype space (PCA 2D): points colored by true rating; prototypes are X")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(True, alpha=0.2)

# Legend (manual)
handles = [
    plt.Line2D([0], [0], marker="o", color="w", label=f"true_rating={int(v) if float(v).is_integer() else v}", markerfacecolor=pal[v], markersize=8)
    for v in distinct_true
]
handles.append(plt.Line2D([0], [0], marker="X", color="black", label="prototype", linestyle="None", markersize=10))
ax.legend(handles=handles, loc="best", frameon=True)

plt.show()

### 4) Closeness summaries

- Distribution of top cosine similarity
- Confusion-style table of true rating vs predicted rating (counts)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(top_sim, bins=30, alpha=0.85, color="#4c78a8")
ax.set_title("Top cosine similarity distribution")
ax.set_xlabel("top_cosine_similarity")
ax.set_ylabel("count")
ax.grid(True, alpha=0.2)
plt.show()

In [ ]:
ct = pd.crosstab(
    pd.Series(true_rating, name="true_rating"),
    pd.Series(pred_rating, name="predicted_rating"),
)
ct

In [ ]:
# Optional: show a few highest/lowest confidence examples
pred_df_sorted = pred_df.sort_values("top_cosine_similarity", ascending=False).reset_index(drop=True)

display_cols = [
    "user_id",
    "asin",
    "true_rating",
    "predicted_rating",
    "top_cosine_similarity",
]

print("Most confident:")
pred_df_sorted.loc[:9, display_cols]

In [ ]:
print("Least confident:")
pred_df_sorted.loc[max(0, len(pred_df_sorted) - 10):, display_cols]